# Explainable Customer Churn Prediction using Apache Spark and Explainable AI
## Big Data Analytics, Spark MLlib Pipelines, and SHAP Interpretability

This notebook demonstrates the end-to-end distributed data processing, exploratory analysis, machine learning modeling, and explainability workflows.

In [ ]:
# 1. Environment Setup & Library Imports
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

print("All libraries imported successfully!")

In [ ]:
# 2. Initialize Apache Spark Session
spark = SparkSession.builder \
    .appName("ExplainableCustomerChurn_EDA") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"Spark Session Active! Version: {spark.version}")

In [ ]:
# 3. Load Telco Dataset into Spark DataFrame
data_path = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
df_spark = spark.read.csv(data_path, header=True, inferSchema=True)

print(f"Rows: {df_spark.count()}, Columns: {len(df_spark.columns)}")
df_spark.printSchema()

In [ ]:
# 4. Spark SQL Data Cleaning & Transformation
df_spark.createOrReplaceTempView("telco")

# TotalCharges cleaning & Tenure Bucketing via Spark SQL
cleaned_df = spark.sql("""
    SELECT 
        *,
        CASE 
            WHEN Churn = 'Yes' THEN 1 
            ELSE 0 
        END AS Churn_Numeric,
        CASE 
            WHEN tenure <= 12 THEN 'New'
            WHEN tenure <= 24 THEN 'Developing'
            WHEN tenure <= 48 THEN 'Stable'
            ELSE 'Loyal'
        END AS tenure_bucket,
        CASE 
            WHEN Contract = 'Month-to-month' THEN 3
            WHEN Contract = 'One year' THEN 2
            ELSE 1
        END AS contract_risk_score
    FROM telco
""")

cleaned_df.select("customerID", "tenure", "tenure_bucket", "Contract", "contract_risk_score", "Churn_Numeric").show(5)

In [ ]:
# 5. Churn Rate Breakdown by Contract Duration
contract_churn = spark.sql("""
    SELECT 
        Contract,
        COUNT(*) AS TotalCustomers,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS ChurnedCount,
        ROUND(AVG(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0.0 END) * 100, 2) AS ChurnRatePct,
        ROUND(SUM(MonthlyCharges), 2) AS TotalMonthlySpend
    FROM telco
    GROUP BY Contract
    ORDER BY ChurnRatePct DESC
""")

contract_churn.show()

In [ ]:
# 6. Convert to Pandas for Visualizations & SHAP
pdf = cleaned_df.toPandas()
pdf["TotalCharges"] = pd.to_numeric(pdf["TotalCharges"].astype(str).str.strip(), errors="coerce").fillna(pdf["MonthlyCharges"] * pdf["tenure"])

plt.figure(figsize=(10, 5), dpi=150)
sns.countplot(data=pdf, x="Contract", hue="Churn", palette="Set2")
plt.title("Customer Churn Distribution by Contract Type", fontsize=14, fontweight="bold")
plt.show()